In [ ]:
workspace_id = ""
bronze_lakehouse_id = ""
destination_lakehouse_id = ""
destination_lakehouse_path = ""
expected_capability_version = "1.0.0"
expected_installed_capabilities =  {}
prefix = ""
deployment_environment = ""

StatementMeta(, 68d43252-d1bf-4fef-8fda-abe43343a9e1, 21, Finished, Available, Finished)

In [12]:
%pip install unittest-xml-reporting

StatementMeta(, 68d43252-d1bf-4fef-8fda-abe43343a9e1, 18, Finished, Available, Finished)


[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [2]:
import xmlrunner

StatementMeta(, 3308407d-eaeb-44ee-927a-435e6e433003, 9, Finished, Available, Finished)

In [ ]:
import io
import requests
import unittest
import logging
import sempy.fabric as fabric
import json
import pandas as pd
import xmlrunner
import sempy.fabric as fabric

EXPECTED_SILVER_TABLES = [
    'Account', 'ActivityDefinition', 'AdverseEvent', 'AllergyIntolerance', 
    'ApplicationCommunication', 'Appointment', 'AppointmentResponse', 
    'AuditEvent', 'Basic', 'Binary', 'BiologicallyDerivedProduct', 
    'BodyStructure', 'Bundle', 'CapabilityStatement', 'CarePlan', 
    'CareTeam', 'CatalogEntry', 'ChargeItem', 'ChargeItemDefinition', 
    'Claim', 'ClaimResponse', 'ClinicalImpression', 'CodeSystem', 
    'Communication', 'CommunicationMethod', 'CommunicationRequest', 
    'CommunicationStatus', 'CommunicationStatusType', 'CompartmentDefinition', 
    'Composition', 'ConceptMap', 'Condition', 'Consent', 'Contract', 
    'Coverage', 'CoverageEligibilityRequest', 'CoverageEligibilityResponse', 
    'DaxTranscripts', 'DetectedIssue', 'Device', 'DeviceDefinition', 
    'DeviceMetric', 'DeviceRequest', 'DeviceUseStatement', 'DiagnosticReport', 
    'DocumentManifest', 'DocumentReference', 'DocumentReferenceContent', 
    'EffectEvidenceSynthesis', 'EmailCommunication', 'EmbeddingEnrichments', 
    'Encounter', 'Endpoint', 'EnrollmentRequest', 'EnrollmentResponse', 
    'EpisodeOfCare', 'EventDefinition', 'Evidence', 'EvidenceVariable', 
    'ExampleScenario', 'ExplanationOfBenefit', 'FamilyMemberHistory', 
    'Flag', 'Goal', 'GraphDefinition', 'Group', 'GuidanceResponse', 
    'HealthcareService', 'IdmCommunication', 'IdmPatient', 'ImagingMetastore', 
    'ImagingStudy', 'Immunization', 'ImmunizationEvaluation', 
    'ImmunizationRecommendation', 'ImplementationGuide', 'InsurancePlan', 
    'Invoice', 'Language', 'Library', 'Linkage', 'List', 'Location', 
    'MarketingCampaignTask', 'MarketingCampaignTaskCommunication', 
    'MarketingCampaignTaskRelatedParty', 'MarketingCampaignTaskStatus', 
    'MarketingCampaignTaskType', 'Measure', 'MeasureReport', 'Media', 
    'Medication', 'MedicationAdministration', 'MedicationDispense', 
    'MedicationKnowledge', 'MedicationRequest', 'MedicationStatement', 
    'MedicinalProduct', 'MedicinalProductAuthorization', 
    'MedicinalProductContraindiction', 'MedicinalProductIndictation', 
    'MedicinalProductIngredient', 'MedicinalProductInteraction', 
    'MedicinalProductManufactured', 'MedicinalProductPackaged', 
    'MedicinalProductPharmaceutical', 'MedicinalProductUndesirableEffect', 
    'MessageDefinition', 'MessageHeader', 'MolecularSequence', 
    'NamingSystem', 'NutritionOrder', 'ObjectEnrichments', 'Observation', 
    'ObservationDefinition', 'OperationDefinition', 'OperationOutcome', 
    'Organization', 'OrganizationAffiliation', 'Parameters', 'Party', 
    'PartyType', 'Patient', 'PaymentNotice', 'PaymentReconciliation', 
    'Person', 'PlanDefinition', 'Practitioner', 'PractitionerRole', 
    'Procedure', 'Provenance', 'Questionnaire', 'QuestionnaireResponse', 
    'RelatedCommunication', 'RelatedMarketingCampaignTask', 'RelatedPerson', 
    'RequestGroup', 'ResearchDefinition', 'ResearchElementDefinition', 
    'ResearchStudy', 'ResearchSubject', 'RiskAssessment', 
    'RiskEvidenceSynthesis', 'Schedule', 'SearchParameter', 
    'Segmentation2DEnrichments', 'Segmentation3DEnrichments', 
    'ServiceRequest', 'Slot', 'SmsCommunication', 'SocialDeterminant', 
    'SocialDeterminantCategory', 'SocialDeterminantDataSetMetadata', 
    'SocialDeterminantSubCategory', 'SoftwareProduct', 
    'SoftwareProductVersion', 'Specimen', 'SpecimenDefinition', 
    'StructureDefinition', 'StructureMap', 'Subscription', 'Substance', 
    'SubstanceNucleicAcid', 'SubstancePolymer', 'SubstanceProtein', 
    'SubstanceReferenceInformation', 'SubstanceSourceMaterial', 
    'SubstanceSpecification', 'SupplyDelivery', 'SupplyRequest', 'Task', 
    'TaskPartyRelationshipType', 'TaskRelationshipType', 'TaskStatusType', 
    'TerminologyCapabilities', 'TestReport', 'TestScript', 'TextEnrichments', 
    'UnitOfMeasure', 'ValueSet', 'VerificationResult', 'VisionPrescription', 
    'ZipToFipsMapping'
]

PBI_GLOBAL_SERVICE_ENDPOINTS = {
    "public": "https://api.powerbi.com/",
    "fairfax": "https://api.powerbigov.us",
    "mooncake": "https://api.powerbi.cn",
    "blackforest": "https://app.powerbi.de",
    "msit": "https://api.powerbi.com/",
    "prod": "https://api.powerbi.com/",
    "int3": "https://biazure-int-edog-redirect.analysis-df.windows.net/",
    "dxt": "https://powerbistagingapi.analysis.windows.net/",
    "edog": "https://biazure-int-edog-redirect.analysis-df.windows.net/",
    "dev": "https://onebox-redirect.analysis.windows-int.net/",
    "console": "http://localhost:5001/",
    "daily": "https://dailyapi.powerbi.com/",
}

DEFAULT_GLOBAL_SERVICE_ENDPOINT = "https://api.powerbi.com/"
FETCH_CLUSTER_DETAIL_URI = "powerbi/globalservice/v201606/clusterDetails"

class CapabilityDeploymentTests(unittest.TestCase):

    def __init__(self, methodName='runTest', spark=None, env = "msit", workspace_id = None, bronze_lakehouse_id = None, expected_installed_capabilities=[], prefix="healthcare1"):
        super().__init__(methodName)
        logging.basicConfig()
        self.client = fabric.FabricRestClient()
        self.logger = logging.getLogger("LOG")
        self.spark = spark
        self.env = env
        self.workspace_id = workspace_id
        self.bronze_lakehouse_id = bronze_lakehouse_id
        self.expected_capability_version = expected_capability_version
        self.expected_installed_capabilities = expected_installed_capabilities
        self.prefix = prefix
        

    def setUp(self):
        self.token = mssparkutils.credentials.getToken('https://analysis.windows.net/powerbi/api')
        self.runtime_context = mssparkutils.runtime.context
        self.workspace_id = self.runtime_context["currentWorkspaceId"]
        self.capacity_id = self.client.get(f"v1/workspaces/{self.workspace_id}").json()["capacityId"]
        self.solution_id = self.client.get(f"v1/workspaces/{self.workspace_id}/items?type=Healthcaredatasolution").json()["value"][0]["id"]
        self.installed_capabilities = self.get_installed_capabilities(self.env, self.capacity_id, self.workspace_id, self.solution_id, self.token)
        self.available_capabilities = self.get_all_capabilities(self.env, self.capacity_id, self.workspace_id, self.solution_id, self.token, "1.0.0")

    # Helper methods
    def get_shared_host(self, env: str, token: str):
        
        url = PBI_GLOBAL_SERVICE_ENDPOINTS.get(env, DEFAULT_GLOBAL_SERVICE_ENDPOINT) + FETCH_CLUSTER_DETAIL_URI
        headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
        resp = requests.get(url, headers=headers)
        resp_body = json.loads(resp.content)
        return resp_body["clusterUrl"]

    def get_request_info(self, env, capacity_id, workspace_id, solution_id, shared_host = None, token = None):
        mwc_token_details = self.get_hds_mwc_token_details(env, capacity_id, workspace_id, workload_type="dmh", shared_host=shared_host, token=token)
        
        mwc_token = mwc_token_details["Token"]
        target_uri = mwc_token_details["TargetUriHost"]
        
        return[
            target_uri, 
            {
                'Authorization': f'MwcToken {mwc_token}',
                'Content-Type': 'application/json',
                'x-ms-workload-resource-moniker': solution_id
            }
        ]

    def get_hds_mwc_token_details(self, env: str, capacity_id: str, workspace_id: str, workload_type: str = "dmh", shared_host = None, token: str = ""):

        if shared_host is None:
            shared_host = self.get_shared_host(env, token)

        payload = {
            "capacityObjectId": capacity_id,
            "workspaceObjectId": workspace_id,
            "workloadType": workload_type,
        }

        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }

        url = f"{shared_host}/metadata/v201606/generatemwctokenv2"
        response = requests.post(url=url, data=json.dumps(payload), headers=headers)
        mwc_token_details = json.loads(response.content)
        return mwc_token_details

    def get_installed_capabilities(self, env, capacity_id: str, workspace_id: str, solution_id: str, token):
        
        shared_host = self.get_shared_host(env, token)
        [target_uri, headers] = self.get_request_info(env, capacity_id, workspace_id, solution_id, shared_host, token)
        endpoint = "https://{}/webapi/capacities/{}/workloads/dmh/DMHService/automatic/artifacts/{}/capabilities"
        url = endpoint.format(target_uri, capacity_id, solution_id)
        response = requests.get(url=url, headers=headers)
        payload = response.json()
        # print(payload)
        return payload

    def get_all_capabilities(self, env, capacity_id, workspace_id, solution_id, token, version = "1.0.0"):

        shared_host = self.get_shared_host(env, token)
        [target_uri, headers] = self.get_request_info(env, capacity_id, workspace_id, solution_id, shared_host, token)
        endpoint = "https://{}/webapi/capacities/{}/workloads/dmh/DMHService/automatic/capabilities?artifactVersion={}"
        url = endpoint.format(target_uri, capacity_id, version)
        response = requests.get(url=url, headers=headers)
        return response.json()

    def get_workspace_items_by_type(self, workspace_id, item_type):
        client = fabric.FabricRestClient()

        response = client.get(f"v1/workspaces/{workspace_id}/items?type={item_type}")
        items = response.json()['value']

        item_dict = {}
        for item in items:
            item_dict[item['displayName']] = item
        
        return item_dict

    # Tests
    def test_installed_capabilities_match_expected(self):
        installed_capabilities_keys = [c["name"] for c in self.installed_capabilities]
        for key in self.expected_installed_capabilities:
            self.assertIn(key, installed_capabilities_keys)

    def test_capabilities_deployed_successfully(self):
        capability_provisioning_states = [c["provisionState"] for c in self.installed_capabilities]
        for capability_provisioning_state in capability_provisioning_states:
            self.assertEqual("Active", capability_provisioning_state)

    def test_deployed_capabilities_have_expected_version(self):
        print(self.installed_capabilities)

        capability_versions = { c["name"] : c["version"] for c in self.installed_capabilities }
        for expected_capability in self.expected_installed_capabilities.keys():
            self.assertTrue(expected_capability in capability_versions)
            self.assertEqual(self.expected_installed_capabilities[expected_capability], capability_versions[expected_capability])

    def test_lakehouses_deployed_succesfully(self):
        
        deployed_lakehouses = self.get_workspace_items_by_type(self.workspace_id, "Lakehouse")

        self.assertTrue(prefix + "_msft_bronze" in deployed_lakehouses)
        self.assertTrue(prefix + "_msft_silver" in deployed_lakehouses)
        self.assertTrue(prefix + "_msft_admin" in deployed_lakehouses)

    def test_notebooks_deployed_successfully(self):
        
        deployed_notebooks = self.get_workspace_items_by_type(self.workspace_id, "Notebook")

        self.assertTrue(prefix + "_msft_config_notebook" in deployed_notebooks)
        self.assertTrue(prefix + "_msft_raw_process_movement" in deployed_notebooks)
        self.assertTrue(prefix + "_msft_fhir_ndjson_bronze_ingestion" in deployed_notebooks)
        self.assertTrue(prefix + "_msft_bronze_silver_flatten" in deployed_notebooks)
        self.assertTrue(prefix + "_msft_fhir_flattening_sample" in deployed_notebooks)
        self.assertTrue(prefix + "_msft_fhir_ndjson_bronze_ingestion" in deployed_notebooks)

    def test_datapipeline_deployed_successfully(self):

        deployed_datapipelines = self.get_workspace_items_by_type(self.workspace_id, "DataPipeline")
        self.assertTrue(prefix + "_msft_clinical_data_foundation_ingestion" in deployed_datapipelines)

    def test_environment_deployed_successfully(self):
        
        deployed_environments = self.get_workspace_items_by_type(self.workspace_id, "Environment")
        self.assertTrue(prefix + "_environment" in deployed_environments)


    def test_validate_bronze_lakehouse_folder_structure(self):
        bronze_lakehouse_path = f"abfss://{self.workspace_id}@{self.env}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}"
        self.assertTrue(mssparkutils.fs.exists(f"{bronze_lakehouse_path}/Files/External"))
        self.assertTrue(mssparkutils.fs.exists(f"{bronze_lakehouse_path}/Files/Failed"))
        self.assertTrue(mssparkutils.fs.exists(f"{bronze_lakehouse_path}/Files/Inventory"))
        self.assertTrue(mssparkutils.fs.exists(f"{bronze_lakehouse_path}/Files/ReferenceData"))
    
    def test_validate_bronze_lakehouse_has_ingest_path(self):
        bronze_lakehouse_path = f"abfss://{self.workspace_id}@{self.env}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}"
        self.assertTrue(mssparkutils.fs.exists(f"{bronze_lakehouse_path}/Files/Ingest/Clinical/FHIR-NDJSON/FHIR-HDS"))
    
    def test_validate_bronze_lakehouse_has_sample_data_path(self):
        bronze_lakehouse_path = f"abfss://{self.workspace_id}@{self.env}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}"
        self.assertTrue(mssparkutils.fs.exists(f"{bronze_lakehouse_path}/Files/SampleData/Clinical/FHIR-NDJSON/FHIR-HDS"))
    
    def test_validate_bronze_lakehouse_has_process_path(self):
        bronze_lakehouse_path = f"abfss://{self.workspace_id}@{self.env}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}"
        self.assertTrue(mssparkutils.fs.exists(f"{bronze_lakehouse_path}/Files/Process/Clinical/FHIR-NDJSON/FHIR-HDS"))
    
    def test_validate_bronze_tables_exist(self):
        lakehouse_root = f"abfss://{self.workspace_id}@{self.env}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}"
        self.assertTrue(mssparkutils.fs.exists(f"{lakehouse_root}/Tables/ClinicalFhir"))

    def test_validate_silver_tables_exist(self):
        
        deployed_lakehouses = self.get_workspace_items_by_type(self.workspace_id, "Lakehouse")

        silver_lakehouse = None
        for lh in deployed_lakehouses.keys():
            if "silver" in str(lh).lower():
                silver_lakehouse = deployed_lakehouses[lh]

        self.assertTrue(silver_lakehouse is not None)

        silver_lakehouse_id = silver_lakehouse["id"]

        silver_lakehouse_path = f"abfss://{workspace_id}@{self.env}-onelake.dfs.fabric.microsoft.com/{silver_lakehouse_id}/Tables"
        print(silver_lakehouse_path)
        deployed_silver_tables = [str(p.path).split("/")[-1] for p in mssparkutils.fs.ls(silver_lakehouse_path)]

        self.assertEqual(len(EXPECTED_SILVER_TABLES), len(deployed_silver_tables))

        for table in EXPECTED_SILVER_TABLES:
            self.assertTrue(table in deployed_silver_tables)
        

    def test_validate_admin_lakehouse_folder_struct(self):
        
        expected_system_config_files = ['deploymentParametersConfiguration.json', 'fileOrchestrationConfig.json', 'validationConfig.json']
        deployed_lakehouses = self.get_workspace_items_by_type(self.workspace_id, "Lakehouse")

        for lh in deployed_lakehouses.keys():
            if "admin" in str(lh).lower():
                admin_lakehouse = deployed_lakehouses[lh]

        self.assertTrue(admin_lakehouse is not None)

        admin_lakehouse_id = admin_lakehouse["id"]
        admin_lakehouse_path = f"abfss://{self.workspace_id}@{self.env}-onelake.dfs.fabric.microsoft.com/{admin_lakehouse_id}/Files/system-configurations"

        self.assertTrue(mssparkutils.fs.exists(admin_lakehouse_path))

        system_config_files = [str(p.path).split("/")[-1] for p in mssparkutils.fs.ls(admin_lakehouse_path)]

        self.assertTrue(len(system_config_files) == len(expected_system_config_files))

        for config in expected_system_config_files:
            self.assertTrue(config in system_config_files)


    def test_validate_admin_contains_business_events_table(self):
        
        deployed_lakehouses = self.get_workspace_items_by_type(self.workspace_id, "Lakehouse")

        admin_lakehouse = None
        for lh in deployed_lakehouses.keys():
            if "admin" in str(lh).lower():
                admin_lakehouse = deployed_lakehouses[lh]

        self.assertTrue(admin_lakehouse is not None)

        admin_lakehouse_id = admin_lakehouse["id"]
        admin_lakehouse_path = f"abfss://{self.workspace_id}@{self.env}-onelake.dfs.fabric.microsoft.com/{admin_lakehouse_id}/Tables"

        deployed_admin_tables = [str(p.path).split("/")[-1] for p in mssparkutils.fs.ls(admin_lakehouse_path)]

        self.assertTrue("BusinessEvents" in deployed_admin_tables)


    def test_validate_deployment_parameter_activity_id_match_notebooks(self):

        deployed_lakehouses = self.get_workspace_items_by_type(self.workspace_id, "Lakehouse")

        for lh in deployed_lakehouses.keys():
            if "admin" in str(lh).lower():
                admin_lakehouse = deployed_lakehouses[lh]

        admin_lakehouse_id = admin_lakehouse["id"]
        deployment_params_file_path = f"abfss://{workspace_id}@{self.env}-onelake.dfs.fabric.microsoft.com/{admin_lakehouse_id}/Files/system-configurations/deploymentParametersConfiguration.json"
        deployment_params_data = json.loads(spark.read.json(deployment_params_file_path, multiLine=True).toJSON().collect()[0])

        configured_activities = {}
        for activity in deployment_params_data["activities"].keys():
            configured_activities[activity] = deployment_params_data["activities"][activity]['name']

        deployed_notebooks = self.get_workspace_items_by_type(self.workspace_id, "Notebook")

        deployed_notebooks_dict = {}
        for nb_name, nb_data in deployed_notebooks.items():
            deployed_notebooks_dict[nb_data["id"]] = nb_data["displayName"]

        print(deployed_notebooks_dict)

        for activity in configured_activities.keys():
            print(activity)
            self.assertIn(activity, deployed_notebooks_dict)
            self.assertTrue(configured_activities[activity] == deployed_notebooks_dict[activity])

    def test_available_capabilities_match_expected(self):

        capability_keys = []
        capabilities = self.get_all_capabilities(self.env, self.capacity_id, self.workspace_id, self.solution_id, self.token, "1.0.0")

        for c in capabilities:
            capability_keys.append(c["name"])

        expected_capabilities = [
                    'relational-fhir-data-foundations',
                    'fhir-data-ingestion',
                    'dicom-data-ingestion',
                    'patient-outreach-analytics-advanced',
                    'clinical-notes-enrichment',
                    'omop-analytics',
                    'patient-outreach-analytics',
                    'sample-data',
                    'healthcare-data-explorer',
                    'social-determinants-of-health',
                    'claims-data-ingestion',
                    'care-management-analytics',
                    'dax-conversational-data-enrichments'
                ]
  
    
        
        print("Available capabilities for v1.0.0")
        for c in expected_capabilities:         
            print(c)
            self.assertIn(c, capability_keys)

def run_tests_and_write_output(spark):
    
    # Run the tests
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(CapabilityDeploymentTests)
    print([test.__str__().split(" ")[0] for test in suite])

    for test in suite:
        print(test.__str__().split(" ", maxsplit=1)[0])
        test.spark = spark
        test.env = deployment_environment
        test.workspace_id = workspace_id
        test.bronze_lakehouse_id = bronze_lakehouse_id
        test.expected_capability_version = expected_capability_version
        test.expected_installed_capabilities = expected_installed_capabilities
        test.prefix = prefix

    write_stream = io.BytesIO()
    xmlrunner.XMLTestRunner(output=write_stream).run(suite)

    xml_output = write_stream.getvalue().decode('utf-8')

    mssparkutils.fs.put(f"abfss://{workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{destination_lakehouse_id}/{destination_lakehouse_path}", xml_output, overwrite=True)

run_tests_and_write_output(spark)

StatementMeta(, 3308407d-eaeb-44ee-927a-435e6e433003, 21, Finished, Available, Finished)


Running tests...
----------------------------------------------------------------------
F...
FAIL [4.666s]: test_available_capabilities_match_expected (__main__.CapabilityDeploymentTests.test_available_capabilities_match_expected)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_14781/3932477369.py", line 156, in test_available_capabilities_match_expected
    self.assertTrue(False)
AssertionError: False is not true

----------------------------------------------------------------------
Ran 4 tests in 13.623s

FAILED (failures=1)

Generating XML reports...
